# Nível 1 — Carregamento e inspeção inicial

Nesta etapa, observamos os dados brutos sem remover, preencher, converter ou sinalizar registros.

In [1]:
import json

import pandas as pd

In [2]:
caminho_dados = '../dados/dados_nivel_1.json'

with open(caminho_dados, encoding='utf-8') as arquivo:
    dados_brutos = json.load(arquivo)

In [3]:
taxa_cambio_usd_brl = dados_brutos['taxa_cambio_usd_brl']
taxa_cambio_usd_brl

5.4

In [4]:
df_operacoes = pd.DataFrame(dados_brutos['operacoes'])

In [5]:
df_operacoes.head()

        id cliente_id  ...         contraparte  observacao
0  OP-0001    CLI-A-1  ...  Alfa Comercio LTDA            
1  OP-0002    CLI-A-1  ...  Alfa Comercio LTDA            
2  OP-0003    CLI-A-1  ...    Beta Servicos ME            
3  OP-0004    CLI-A-1  ...  Gama Distribuidora            
4  OP-0005    CLI-A-2  ...   Delta Transportes            

[5 rows x 9 columns]

In [6]:
quantidade_linhas, quantidade_colunas = df_operacoes.shape
print(f'Linhas: {quantidade_linhas}')
print(f'Colunas: {quantidade_colunas}')

Linhas: 20
Colunas: 9


In [7]:
df_operacoes.columns.tolist()

['id', 'cliente_id', 'data', 'valor', 'moeda', 'canal', 'tipo', 'contraparte', 'observacao']

In [8]:
df_operacoes.dtypes

id               str
cliente_id       str
data             str
valor          int64
moeda            str
canal            str
tipo             str
contraparte      str
observacao       str
dtype: object

In [9]:
df_operacoes.isna().sum()

id             0
cliente_id     0
data           1
valor          0
moeda          0
canal          0
tipo           0
contraparte    0
observacao     0
dtype: int64

In [10]:
ids_duplicados = df_operacoes['id'].duplicated(keep=False)
print(f'Linhas com ID duplicado: {ids_duplicados.sum()}')
df_operacoes.loc[ids_duplicados, ['id']]

Linhas com ID duplicado: 2


        id
6  OP-0007
9  OP-0007

## Limpeza dos dados

A seguir, a base bruta é preservada e criamos uma versão tratada somente para remover duplicatas exatas e converter o tipo da coluna de data.

In [11]:
comparacao_op_0007 = df_operacoes.loc[df_operacoes['id'] == 'OP-0007'].reset_index(drop=True)
sao_duplicatas_exatas = (
    len(comparacao_op_0007) == 2
    and comparacao_op_0007.duplicated(keep=False).all()
)
print(f'As duas ocorrências de OP-0007 são duplicatas exatas: {sao_duplicatas_exatas}')
comparacao_op_0007.T

As duas ocorrências de OP-0007 são duplicatas exatas: True


                                 0                      1
id                         OP-0007                OP-0007
cliente_id                 CLI-A-3                CLI-A-3
data                    2026-03-05             2026-03-05
valor                        17200                  17200
moeda                          BRL                    BRL
canal                          pix                    pix
tipo         transferencia_enviada  transferencia_enviada
contraparte    Epsilon Consultoria    Epsilon Consultoria
observacao                                               

In [12]:
linhas_antes_remocao = len(df_operacoes)
df_operacoes_tratado = df_operacoes.drop_duplicates()
linhas_depois_remocao = len(df_operacoes_tratado)

print(f'Linhas antes da remoção: {linhas_antes_remocao}')
print(f'Linhas depois da remoção: {linhas_depois_remocao}')

Linhas antes da remoção: 20
Linhas depois da remoção: 19


In [13]:
df_operacoes_tratado['data'] = pd.to_datetime(df_operacoes_tratado['data'])
df_operacoes_tratado['data'].dtype

dtype('<M8[us]')

In [14]:
df_operacoes_tratado.isna().sum()

id             0
cliente_id     0
data           1
valor          0
moeda          0
canal          0
tipo           0
contraparte    0
observacao     0
dtype: int64

### Decisão de limpeza

A duplicata exata de `OP-0007` foi removida para evitar que uma mesma operação seja contada duas vezes nas análises posteriores. Mantivemos uma ocorrência para preservar a operação legítima registrada na base.

O registro sem `data` foi mantido porque os demais campos ainda podem contribuir para análises que não dependem de data, como volume total e contagens gerais. Esse registro não poderá participar de regras que exigem agrupamento temporal, pois não inventamos nem preenchemos uma data ausente.

## Normalização monetária

Nesta etapa, criamos uma coluna em uma única moeda de referência, sem alterar os valores e as moedas originalmente registrados.

In [15]:
df_operacoes_tratado['moeda'].unique()

<StringArray>
['BRL', 'USD']
Length: 2, dtype: str

In [16]:
operacoes_usd_antes = df_operacoes_tratado.loc[
    df_operacoes_tratado['moeda'] == 'USD',
    ['id', 'cliente_id', 'valor', 'moeda']
]
operacoes_usd_antes

         id cliente_id  valor moeda
13  OP-0013    CLI-A-4  12000   USD

In [17]:
df_operacoes_tratado['valor_brl'] = df_operacoes_tratado['valor'].astype(float)

mascara_usd = df_operacoes_tratado['moeda'] == 'USD'
df_operacoes_tratado.loc[mascara_usd, 'valor_brl'] = (
    df_operacoes_tratado.loc[mascara_usd, 'valor'] * taxa_cambio_usd_brl
)

In [18]:
operacoes_usd_depois = df_operacoes_tratado.loc[
    mascara_usd,
    ['id', 'cliente_id', 'valor', 'moeda', 'valor_brl']
]
operacoes_usd_depois

         id cliente_id  valor moeda  valor_brl
13  OP-0013    CLI-A-4  12000   USD    64800.0

In [19]:
valor_brl_preenchido = df_operacoes_tratado['valor_brl'].notna().all()
print(f'Todas as operações possuem valor_brl preenchido: {valor_brl_preenchido}')

Todas as operações possuem valor_brl preenchido: True


### Decisão de normalização monetária

Normalizamos as moedas para que valores em BRL e USD possam ser comparados e somados corretamente em análises futuras. A taxa usada é a do próprio arquivo, pois ela foi fornecida como referência fixa para este desafio e mantém o resultado reproduzível.

As colunas originais `valor` e `moeda` foram mantidas para preservar a informação registrada na operação. A coluna derivada `valor_brl` guarda somente o valor normalizado, permitindo rastrear como cada resultado foi obtido.

## Agregações iniciais

`groupby` é como separar uma planilha em pequenas pilhas de linhas que têm algo em comum. Primeiro formamos uma pilha para cada cliente ou canal; depois calculamos um resultado para cada pilha. As agregações abaixo usam a base tratada e o valor já normalizado em BRL.

In [20]:
volume_total_por_cliente = (
    df_operacoes_tratado.groupby('cliente_id')['valor_brl']
    .sum()
    .rename('volume_total_brl')
    .reset_index()
)
volume_total_por_cliente

  cliente_id  volume_total_brl
0    CLI-A-1           57500.0
1    CLI-A-2           52900.0
2    CLI-A-3           48500.0
3    CLI-A-4           79500.0
4    CLI-A-5           16900.0
5    CLI-A-6           10200.0

In [21]:
quantidade_operacoes_por_canal = (
    df_operacoes_tratado.groupby('canal')
    .size()
    .rename('quantidade_operacoes')
    .reset_index()
)
quantidade_operacoes_por_canal

     canal  quantidade_operacoes
0   boleto                     3
1   cartao                     2
2  especie                     1
3      pix                     8
4      ted                     5

## Regra 1 — Fracionamento

A regra avalia cada combinação de cliente e data. Um grupo é sinalizado quando tem pelo menos três operações, soma acima de R$ 50.000,00 e nenhuma operação individual igual ou acima de R$ 20.000,00. Operações sem data ficam fora deste agrupamento temporal.

In [22]:
MINIMO_OPERACOES_FRACIONAMENTO = 3
LIMITE_SOMA_FRACIONAMENTO = 50_000
LIMITE_VALOR_INDIVIDUAL = 20_000

operacoes_com_data = df_operacoes_tratado.loc[
    df_operacoes_tratado['data'].notna()
].copy()

resumo_fracionamento = (
    operacoes_com_data.groupby(['cliente_id', 'data'], as_index=False)
    .agg(
        quantidade_operacoes=('id', 'size'),
        soma_valor_brl=('valor_brl', 'sum'),
        maior_valor_individual_brl=('valor_brl', 'max'),
    )
)

resumo_fracionamento['flag_fracionamento'] = (
    (resumo_fracionamento['quantidade_operacoes'] >= MINIMO_OPERACOES_FRACIONAMENTO)
    & (resumo_fracionamento['soma_valor_brl'] > LIMITE_SOMA_FRACIONAMENTO)
    & (resumo_fracionamento['maior_valor_individual_brl'] < LIMITE_VALOR_INDIVIDUAL)
)
resumo_fracionamento

   cliente_id       data  ...  maior_valor_individual_brl  flag_fracionamento
0     CLI-A-1 2026-03-09  ...                     18800.0                True
1     CLI-A-1 2026-03-21  ...                      3300.0               False
2     CLI-A-2 2026-03-14  ...                     27000.0               False
3     CLI-A-3 2026-03-05  ...                     17200.0               False
4     CLI-A-4 2026-03-03  ...                      3800.0               False
5     CLI-A-4 2026-03-11  ...                      5100.0               False
6     CLI-A-4 2026-03-18  ...                      5800.0               False
7     CLI-A-4 2026-03-24  ...                     64800.0               False
8     CLI-A-5 2026-03-07  ...                      2900.0               False
9     CLI-A-5 2026-03-16  ...                      7000.0               False
10    CLI-A-5 2026-03-26  ...                      2700.0               False
11    CLI-A-6 2026-03-12  ...                      8800.0       

In [23]:
df_operacoes_tratado = df_operacoes_tratado.merge(
    resumo_fracionamento[['cliente_id', 'data', 'flag_fracionamento']],
    on=['cliente_id', 'data'],
    how='left',
)
df_operacoes_tratado['flag_fracionamento'] = (
    df_operacoes_tratado['flag_fracionamento'].fillna(False).astype(bool)
)

df_operacoes_tratado.loc[
    df_operacoes_tratado['flag_fracionamento'],
    ['id', 'cliente_id', 'data', 'valor_brl', 'flag_fracionamento'],
]

        id cliente_id       data  valor_brl  flag_fracionamento
0  OP-0001    CLI-A-1 2026-03-09    18100.0                True
1  OP-0002    CLI-A-1 2026-03-09    17300.0                True
2  OP-0003    CLI-A-1 2026-03-09    18800.0                True

In [24]:
validacao_cli_a_1 = resumo_fracionamento.loc[
    (resumo_fracionamento['cliente_id'] == 'CLI-A-1')
    & (resumo_fracionamento['data'] == pd.Timestamp('2026-03-09'))
]
validacao_cli_a_2 = resumo_fracionamento.loc[
    (resumo_fracionamento['cliente_id'] == 'CLI-A-2')
    & (resumo_fracionamento['data'] == pd.Timestamp('2026-03-14'))
]

flag_cli_a_1 = validacao_cli_a_1['flag_fracionamento'].iloc[0]
flag_cli_a_2 = validacao_cli_a_2['flag_fracionamento'].iloc[0]

assert bool(flag_cli_a_1)
assert not bool(flag_cli_a_2)

print(f'CLI-A-1 em 2026-03-09 foi capturado: {flag_cli_a_1}')
print(f'CLI-A-2 em 2026-03-14 não foi capturado: {not flag_cli_a_2}')
pd.concat([validacao_cli_a_1, validacao_cli_a_2])

CLI-A-1 em 2026-03-09 foi capturado: True
CLI-A-2 em 2026-03-14 não foi capturado: True


  cliente_id       data  ...  maior_valor_individual_brl  flag_fracionamento
0    CLI-A-1 2026-03-09  ...                     18800.0                True
2    CLI-A-2 2026-03-14  ...                     27000.0               False

[2 rows x 6 columns]

## Regra 2 — Valor atípico

A mediana é o valor central das operações de um cliente quando elas são ordenadas. Ela costuma representar melhor o padrão normal do que a média quando existe um valor muito alto, pois a média é puxada por esse extremo. Aplicamos a regra somente a clientes com quatro ou mais operações, pois uma amostra menor oferece pouco histórico para comparação.

In [25]:
MINIMO_OPERACOES_VALOR_ATIPICO = 4
MULTIPLICADOR_VALOR_ATIPICO = 5

perfil_valor_por_cliente = (
    df_operacoes_tratado.groupby('cliente_id', as_index=False)
    .agg(
        quantidade_operacoes_cliente=('id', 'size'),
        mediana_valor_brl=('valor_brl', 'median'),
    )
)
perfil_valor_por_cliente

  cliente_id  quantidade_operacoes_cliente  mediana_valor_brl
0    CLI-A-1                             4            17700.0
1    CLI-A-2                             2            26450.0
2    CLI-A-3                             3            16100.0
3    CLI-A-4                             4             5450.0
4    CLI-A-5                             4             3600.0
5    CLI-A-6                             2             5100.0

In [26]:
df_operacoes_tratado = df_operacoes_tratado.merge(
    perfil_valor_por_cliente,
    on='cliente_id',
    how='left',
)
df_operacoes_tratado['limite_valor_atipico_brl'] = (
    MULTIPLICADOR_VALOR_ATIPICO * df_operacoes_tratado['mediana_valor_brl']
)
df_operacoes_tratado['flag_valor_atipico'] = (
    (df_operacoes_tratado['quantidade_operacoes_cliente'] >= MINIMO_OPERACOES_VALOR_ATIPICO)
    & (df_operacoes_tratado['valor_brl'] > df_operacoes_tratado['limite_valor_atipico_brl'])
)

df_operacoes_tratado[
    [
        'id',
        'cliente_id',
        'valor_brl',
        'quantidade_operacoes_cliente',
        'mediana_valor_brl',
        'limite_valor_atipico_brl',
        'flag_valor_atipico',
    ]
]

         id cliente_id  ...  limite_valor_atipico_brl  flag_valor_atipico
0   OP-0001    CLI-A-1  ...                   88500.0               False
1   OP-0002    CLI-A-1  ...                   88500.0               False
2   OP-0003    CLI-A-1  ...                   88500.0               False
3   OP-0004    CLI-A-1  ...                   88500.0               False
4   OP-0005    CLI-A-2  ...                  132250.0               False
5   OP-0006    CLI-A-2  ...                  132250.0               False
6   OP-0007    CLI-A-3  ...                   80500.0               False
7   OP-0008    CLI-A-3  ...                   80500.0               False
8   OP-0009    CLI-A-3  ...                   80500.0               False
9   OP-0010    CLI-A-4  ...                   27250.0               False
10  OP-0011    CLI-A-4  ...                   27250.0               False
11  OP-0012    CLI-A-4  ...                   27250.0               False
12  OP-0013    CLI-A-4  ...           

In [27]:
validacao_cli_a_4 = df_operacoes_tratado.loc[
    df_operacoes_tratado['cliente_id'] == 'CLI-A-4',
    [
        'id',
        'valor_brl',
        'quantidade_operacoes_cliente',
        'mediana_valor_brl',
        'limite_valor_atipico_brl',
        'flag_valor_atipico',
    ],
]
operacao_op_0013 = validacao_cli_a_4.loc[
    validacao_cli_a_4['id'] == 'OP-0013'
].iloc[0]

print(f"OP-0013 — valor_brl: R$ {operacao_op_0013['valor_brl']:,.2f}")
print(f"Quantidade de operações de CLI-A-4: {operacao_op_0013['quantidade_operacoes_cliente']}")
print(f"Mediana: R$ {operacao_op_0013['mediana_valor_brl']:,.2f}")
print(f"Limite de 5 × mediana: R$ {operacao_op_0013['limite_valor_atipico_brl']:,.2f}")
print(f"Flag de valor atípico: {operacao_op_0013['flag_valor_atipico']}")
validacao_cli_a_4

OP-0013 — valor_brl: R$ 64,800.00
Quantidade de operações de CLI-A-4: 4
Mediana: R$ 5,450.00
Limite de 5 × mediana: R$ 27,250.00
Flag de valor atípico: True


         id  valor_brl  ...  limite_valor_atipico_brl  flag_valor_atipico
9   OP-0010     3800.0  ...                   27250.0               False
10  OP-0011     5100.0  ...                   27250.0               False
11  OP-0012     5800.0  ...                   27250.0               False
12  OP-0013    64800.0  ...                   27250.0                True

[4 rows x 6 columns]

## Validação consolidada das regras

Estes testes usam casos concretos conhecidos da base para verificar se a implementação está aderente às regras. Eles ajudam a identificar mudanças acidentais no comportamento esperado.

In [28]:
validacao_fracionamento = resumo_fracionamento.loc[
    ((resumo_fracionamento['cliente_id'] == 'CLI-A-1')
     & (resumo_fracionamento['data'] == pd.Timestamp('2026-03-09')))
    | ((resumo_fracionamento['cliente_id'] == 'CLI-A-2')
       & (resumo_fracionamento['data'] == pd.Timestamp('2026-03-14'))),
    [
        'cliente_id',
        'data',
        'quantidade_operacoes',
        'soma_valor_brl',
        'maior_valor_individual_brl',
        'flag_fracionamento',
    ],
]

caso_positivo_fracionamento = validacao_fracionamento.loc[
    validacao_fracionamento['cliente_id'] == 'CLI-A-1'
].iloc[0]
caso_negativo_fracionamento = validacao_fracionamento.loc[
    validacao_fracionamento['cliente_id'] == 'CLI-A-2'
].iloc[0]

assert caso_positivo_fracionamento['quantidade_operacoes'] == 3
assert caso_positivo_fracionamento['soma_valor_brl'] == 54_200
assert caso_positivo_fracionamento['maior_valor_individual_brl'] == 18_800
assert bool(caso_positivo_fracionamento['flag_fracionamento'])
assert caso_negativo_fracionamento['quantidade_operacoes'] == 2
assert not bool(caso_negativo_fracionamento['flag_fracionamento'])

validacao_fracionamento.T

                                              0                    2
cliente_id                              CLI-A-1              CLI-A-2
data                        2026-03-09 00:00:00  2026-03-14 00:00:00
quantidade_operacoes                          3                    2
soma_valor_brl                          54200.0              52900.0
maior_valor_individual_brl              18800.0              27000.0
flag_fracionamento                         True                False

In [29]:
validacao_valor_atipico = df_operacoes_tratado.loc[
    df_operacoes_tratado['id'] == 'OP-0013',
    [
        'id',
        'cliente_id',
        'valor_brl',
        'quantidade_operacoes_cliente',
        'mediana_valor_brl',
        'limite_valor_atipico_brl',
        'flag_valor_atipico',
    ],
]

caso_op_0013 = validacao_valor_atipico.iloc[0]
assert caso_op_0013['cliente_id'] == 'CLI-A-4'
assert round(caso_op_0013['valor_brl'], 2) == 64_800
assert caso_op_0013['quantidade_operacoes_cliente'] == 4
assert caso_op_0013['mediana_valor_brl'] == 5_450
assert caso_op_0013['limite_valor_atipico_brl'] == 27_250
assert bool(caso_op_0013['flag_valor_atipico'])

validacao_valor_atipico.T

                                   12
id                            OP-0013
cliente_id                    CLI-A-4
valor_brl                     64800.0
quantidade_operacoes_cliente        4
mediana_valor_brl              5450.0
limite_valor_atipico_brl      27250.0
flag_valor_atipico               True

## Parte B — Parecer estruturado com Gemini

O Pandas continua responsável pelos cálculos e flags. Nesta etapa, o Gemini recebe apenas fatos já calculados para redigir um parecer estruturado de apoio à triagem.

In [30]:
import json
import os
import time
from pathlib import Path
from typing import Literal

import httpx
from dotenv import load_dotenv
from google import genai
from google.genai import errors, types
from pydantic import BaseModel, ValidationError

In [31]:
caminho_env = Path('../.env')
load_dotenv(caminho_env)

gemini_api_key = os.getenv('GEMINI_API_KEY')
gemini_model = os.getenv('GEMINI_MODEL')
gemini_configurada = bool(gemini_api_key and gemini_model)

if gemini_configurada:
    print(f'Modelo Gemini configurado: {gemini_model}')
else:
    print('Configure GEMINI_API_KEY e GEMINI_MODEL no arquivo .env para executar a chamada.')

Modelo Gemini configurado: gemini-3.7-flash


### Validação da resposta

Pydantic funciona como um formulário com regras: ele verifica se todos os campos esperados estão presentes e se os tipos e valores permitidos foram respeitados. Assim, uma resposta com JSON inválido, um campo ausente ou um nível de risco fora das opções aceitas é rejeitada.

In [32]:
class ParecerTriagem(BaseModel):
    nivel_risco: Literal['baixo', 'médio', 'alto']
    tipologia_suspeita: str
    red_flags: list[str]
    justificativa: str

In [33]:
cliente_analisado = 'CLI-A-1'
data_evento_fracionamento = pd.Timestamp('2026-03-09')

evento_fracionamento_cli_a_1 = resumo_fracionamento.loc[
    (resumo_fracionamento['cliente_id'] == cliente_analisado)
    & (resumo_fracionamento['data'] == data_evento_fracionamento)
].iloc[0]
volume_total_cli_a_1 = volume_total_por_cliente.loc[
    volume_total_por_cliente['cliente_id'] == cliente_analisado,
    'volume_total_brl',
].iloc[0]

contexto_cli_a_1 = {
    'cliente_id': cliente_analisado,
    'volume_total_transacionado_brl': float(volume_total_cli_a_1),
    'regra_fracionamento': {
        'data_evento': data_evento_fracionamento.strftime('%Y-%m-%d'),
        'quantidade_operacoes': int(evento_fracionamento_cli_a_1['quantidade_operacoes']),
        'soma_valor_brl': float(evento_fracionamento_cli_a_1['soma_valor_brl']),
        'maior_valor_individual_brl': float(
            evento_fracionamento_cli_a_1['maior_valor_individual_brl']
        ),
        'flag_fracionamento': bool(evento_fracionamento_cli_a_1['flag_fracionamento']),
    },
}
contexto_cli_a_1

{'cliente_id': 'CLI-A-1', 'volume_total_transacionado_brl': 57500.0, 'regra_fracionamento': {'data_evento': '2026-03-09', 'quantidade_operacoes': 3, 'soma_valor_brl': 54200.0, 'maior_valor_individual_brl': 18800.0, 'flag_fracionamento': True}}

In [34]:
prompt_a = f"""
Você atua como analista de apoio à triagem de prevenção à lavagem de dinheiro.

A seguir estão fatos determinísticos, já calculados e validados por Python/Pandas.
Não recalcule somas, contagens, limites ou flags. Interprete somente esses fatos.
Não invente informações ausentes.
O parecer é apoio à triagem humana e não uma decisão automática.

Fatos determinísticos:
{json.dumps(contexto_cli_a_1, ensure_ascii=False, indent=2)}

Retorne um parecer com os campos nivel_risco, tipologia_suspeita, red_flags e justificativa.
"""
print(prompt_a)


Você atua como analista de apoio à triagem de prevenção à lavagem de dinheiro.

A seguir estão fatos determinísticos, já calculados e validados por Python/Pandas.
Não recalcule somas, contagens, limites ou flags. Interprete somente esses fatos.
Não invente informações ausentes.
O parecer é apoio à triagem humana e não uma decisão automática.

Fatos determinísticos:
{
  "cliente_id": "CLI-A-1",
  "volume_total_transacionado_brl": 57500.0,
  "regra_fracionamento": {
    "data_evento": "2026-03-09",
    "quantidade_operacoes": 3,
    "soma_valor_brl": 54200.0,
    "maior_valor_individual_brl": 18800.0,
    "flag_fracionamento": true
  }
}

Retorne um parecer com os campos nivel_risco, tipologia_suspeita, red_flags e justificativa.



### Chamada e métricas

Tokens medem partes do texto processado pelo modelo; a latência mede o tempo da chamada. A API pode disponibilizar tokens de entrada, saída e total, que são registrados quando presentes.

In [35]:
parecer_estruturado = None
metricas_chamada_llm = {}
status_chamada_llm = 'não executada'
erro_chamada_llm = None

if not gemini_configurada:
    print('Chamada não executada: variáveis GEMINI_API_KEY e GEMINI_MODEL não configuradas.')
else:
    inicio_chamada = time.perf_counter()
    try:
        cliente_gemini = genai.Client(api_key=gemini_api_key)
        resposta_gemini = cliente_gemini.models.generate_content(
            model=gemini_model,
            contents=prompt_a,
            config=types.GenerateContentConfig(
                response_mime_type='application/json',
                response_schema=ParecerTriagem,
            ),
        )
        parecer_estruturado = ParecerTriagem.model_validate_json(resposta_gemini.text)
        uso = resposta_gemini.usage_metadata
        metricas_chamada_llm = {
            'latencia_segundos': round(time.perf_counter() - inicio_chamada, 3),
            'tokens_entrada': getattr(uso, 'prompt_token_count', None),
            'tokens_saida': getattr(uso, 'candidates_token_count', None),
            'tokens_totais': getattr(uso, 'total_token_count', None),
        }
        status_chamada_llm = 'sucesso'
    except ValidationError as erro:
        metricas_chamada_llm = {
            'latencia_segundos': round(time.perf_counter() - inicio_chamada, 3)
        }
        status_chamada_llm = 'resposta_inválida'
        erro_chamada_llm = str(erro)
    except httpx.TimeoutException as erro:
        metricas_chamada_llm = {
            'latencia_segundos': round(time.perf_counter() - inicio_chamada, 3)
        }
        status_chamada_llm = 'timeout'
        erro_chamada_llm = str(erro)
    except (errors.ClientError, errors.ServerError) as erro:
        metricas_chamada_llm = {
            'latencia_segundos': round(time.perf_counter() - inicio_chamada, 3)
        }
        status_chamada_llm = 'erro_api'
        erro_chamada_llm = str(erro)

In [36]:
if parecer_estruturado is None:
    print(f'Status: {status_chamada_llm}')
    if erro_chamada_llm:
        print(f'Erro: {erro_chamada_llm}')
else:
    print(parecer_estruturado.model_dump_json(indent=2))

{
  "nivel_risco": "alto",
  "tipologia_suspeita": "Fracionamento de operações (Structuring)",
  "red_flags": [
    "Identificação de flag positiva para fracionamento de transações.",
    "Realização de 3 operações em uma única data (2026-03-09), totalizando R$ 54.200,00.",
    "Maior valor individual de R$ 18.800,00 compondo a maior parte do volume total transacionado (R$ 57.500,00)."
  ],
  "justificativa": "O cliente CLI-A-1 apresentou indicativo de fracionamento de valores em 2026-03-09, com três operações que somam R$ 54.200,00 de um volume total de R$ 57.500,00, com valores unitários abaixo de R$ 20.000,00, sugerindo tentativa de burla a limites regulatórios de reporte."
}


In [37]:
metricas_chamada_llm

{'latencia_segundos': 3.315, 'tokens_entrada': 257, 'tokens_saida': 250, 'tokens_totais': 967}

## Prompt 2 — Parecer com restrições adicionais

O resultado do Prompt 1 é preservado abaixo como registro da primeira chamada. O Prompt 2 recebe exatamente os mesmos fatos determinísticos, com instruções adicionais para reduzir inferências não sustentadas pelos dados.

In [38]:
resposta_prompt_1_registrada = {
    'nivel_risco': 'alto',
    'tipologia_suspeita': 'Fracionamento de operações (Structuring)',
    'red_flags': [
        'Identificação de flag positiva para fracionamento de transações.',
        'Realização de 3 operações em uma única data (2026-03-09), totalizando R$ 54.200,00.',
        'Maior valor individual de R$ 18.800,00 compondo a maior parte do volume total transacionado (R$ 57.500,00).',
    ],
    'justificativa': (
        'O cliente CLI-A-1 apresentou indicativo de fracionamento de valores em 2026-03-09, '
        'com três operações que somam R$ 54.200,00 de um volume total de R$ 57.500,00, '
        'com valores unitários abaixo de R$ 20.000,00, sugerindo tentativa de burla a '
        'limites regulatórios de reporte.'
    ),
}
metricas_prompt_1 = {
    'latencia_segundos': 3.315,
    'tokens_entrada': 257,
    'tokens_saida': 250,
    'tokens_totais': 967,
}
parecer_prompt_1 = ParecerTriagem.model_validate(resposta_prompt_1_registrada)

In [39]:
prompt_2 = f"""
Você atua como analista de apoio à triagem de prevenção à lavagem de dinheiro.

Todos os números, flags e limites abaixo são fatos determinísticos já calculados e validados por Python/Pandas.
Não recalcule, não altere e não reinterprete esses valores.

Restrições obrigatórias:
- Não afirme legislação, regulamentação, intenção do cliente ou motivação que não tenham sido fornecidas.
- Não transforme hipótese em fato.
- Cada red flag deve ser diretamente sustentada pelos dados enviados.
- Quando houver interpretação, use linguagem de possibilidade, como 'pode ser compatível com' ou 'merece análise'.
- O parecer é apoio à triagem humana e não é conclusão de lavagem de dinheiro.
- Não invente dados ausentes.

Fatos determinísticos:
{json.dumps(contexto_cli_a_1, ensure_ascii=False, indent=2)}

Retorne um parecer com os campos nivel_risco, tipologia_suspeita, red_flags e justificativa.
"""
print(prompt_2)


Você atua como analista de apoio à triagem de prevenção à lavagem de dinheiro.

Todos os números, flags e limites abaixo são fatos determinísticos já calculados e validados por Python/Pandas.
Não recalcule, não altere e não reinterprete esses valores.

Restrições obrigatórias:
- Não afirme legislação, regulamentação, intenção do cliente ou motivação que não tenham sido fornecidas.
- Não transforme hipótese em fato.
- Cada red flag deve ser diretamente sustentada pelos dados enviados.
- Quando houver interpretação, use linguagem de possibilidade, como 'pode ser compatível com' ou 'merece análise'.
- O parecer é apoio à triagem humana e não é conclusão de lavagem de dinheiro.
- Não invente dados ausentes.

Fatos determinísticos:
{
  "cliente_id": "CLI-A-1",
  "volume_total_transacionado_brl": 57500.0,
  "regra_fracionamento": {
    "data_evento": "2026-03-09",
    "quantidade_operacoes": 3,
    "soma_valor_brl": 54200.0,
    "maior_valor_individual_brl": 18800.0,
    "flag_fracionamento

In [40]:
parecer_prompt_2 = None
metricas_prompt_2 = {}
status_prompt_2 = 'não executada'
erro_prompt_2 = None

if not gemini_configurada:
    print('Chamada não executada: variáveis GEMINI_API_KEY e GEMINI_MODEL não configuradas.')
else:
    inicio_prompt_2 = time.perf_counter()
    try:
        cliente_gemini_prompt_2 = genai.Client(api_key=gemini_api_key)
        resposta_gemini_prompt_2 = cliente_gemini_prompt_2.models.generate_content(
            model=gemini_model,
            contents=prompt_2,
            config=types.GenerateContentConfig(
                response_mime_type='application/json',
                response_schema=ParecerTriagem,
            ),
        )
        parecer_prompt_2 = ParecerTriagem.model_validate_json(resposta_gemini_prompt_2.text)
        uso_prompt_2 = resposta_gemini_prompt_2.usage_metadata
        metricas_prompt_2 = {
            'latencia_segundos': round(time.perf_counter() - inicio_prompt_2, 3),
            'tokens_entrada': getattr(uso_prompt_2, 'prompt_token_count', None),
            'tokens_saida': getattr(uso_prompt_2, 'candidates_token_count', None),
            'tokens_pensamento': getattr(uso_prompt_2, 'thoughts_token_count', None),
            'tokens_totais': getattr(uso_prompt_2, 'total_token_count', None),
        }
        status_prompt_2 = 'sucesso'
    except ValidationError as erro:
        metricas_prompt_2 = {
            'latencia_segundos': round(time.perf_counter() - inicio_prompt_2, 3)
        }
        status_prompt_2 = 'resposta_inválida'
        erro_prompt_2 = str(erro)
    except httpx.TimeoutException as erro:
        metricas_prompt_2 = {
            'latencia_segundos': round(time.perf_counter() - inicio_prompt_2, 3)
        }
        status_prompt_2 = 'timeout'
        erro_prompt_2 = str(erro)
    except (errors.ClientError, errors.ServerError) as erro:
        metricas_prompt_2 = {
            'latencia_segundos': round(time.perf_counter() - inicio_prompt_2, 3)
        }
        status_prompt_2 = 'erro_api'
        erro_prompt_2 = str(erro)

In [41]:
if parecer_prompt_2 is None:
    print(f'Status: {status_prompt_2}')
    if erro_prompt_2:
        print(f'Erro: {erro_prompt_2}')
else:
    print(parecer_prompt_2.model_dump_json(indent=2))

{
  "nivel_risco": "médio",
  "tipologia_suspeita": "Possível fracionamento de valores (smurfing)",
  "red_flags": [
    "Ativação de flag determinística de fracionamento no evento de 2026-03-09",
    "Execução de 3 operações somando R$ 54.200,00 na mesma data",
    "Maior valor individual de R$ 18.800,00 em um volume total transacionado de R$ 57.500,00"
  ],
  "justificativa": "Os dados apontam a ocorrência de 3 transações em 2026-03-09 totalizando R$ 54.200,00 com maior valor individual de R$ 18.800,00, ativando a flag determinística de fracionamento. Este comportamento pode ser compatível com estruturação de operações e merece análise na triagem humana, não representando conclusão definitiva de lavagem de dinheiro."
}


In [42]:
metricas_prompt_2

{'latencia_segundos': 10.373, 'tokens_entrada': 347, 'tokens_saida': 224, 'tokens_pensamento': 617, 'tokens_totais': 1188}

## Comparação objetiva — Prompt 1 × Prompt 2

A comparação abaixo contrasta as respostas com os fatos determinísticos enviados. Os comentários são uma revisão humana do conteúdo; não foram produzidos por LLM.

In [43]:
comparacao_prompts = pd.DataFrame(
    [
        {
            'prompt': 'Prompt 1',
            'nivel_risco': parecer_prompt_1.nivel_risco,
            'tokens_entrada': metricas_prompt_1['tokens_entrada'],
            'tokens_saida': metricas_prompt_1['tokens_saida'],
            'tokens_pensamento': 'não registrado na chamada 1',
            'tokens_totais': metricas_prompt_1['tokens_totais'],
            'latencia_segundos': metricas_prompt_1['latencia_segundos'],
            'aderencia_aos_fatos': 'Usa os números e a flag fornecidos, mas adiciona uma conclusão além do contexto.',
            'inferencias_nao_suportadas': (
                'Afirma possível burla a limites regulatórios e que R$ 18.800,00 compõe a maior parte do volume; '
                'nenhuma dessas afirmações foi fornecida pelos fatos.'
            ),
            'clareza': 'Clara, porém mais assertiva do que os fatos permitem.',
        },
        {
            'prompt': 'Prompt 2',
            'nivel_risco': parecer_prompt_2.nivel_risco,
            'tokens_entrada': metricas_prompt_2['tokens_entrada'],
            'tokens_saida': metricas_prompt_2['tokens_saida'],
            'tokens_pensamento': metricas_prompt_2.get('tokens_pensamento'),
            'tokens_totais': metricas_prompt_2['tokens_totais'],
            'latencia_segundos': metricas_prompt_2['latencia_segundos'],
            'aderencia_aos_fatos': 'Mantém os números e a flag e usa linguagem condicional.',
            'inferencias_nao_suportadas': (
                'A tipologia é apresentada como possibilidade; não há afirmação legal, intenção do cliente ou motivação.'
            ),
            'clareza': 'Direta, delimitada como apoio à triagem humana.',
        },
    ]
)
comparacao_prompts.T

                                                                            0                                                  1
prompt                                                               Prompt 1                                           Prompt 2
nivel_risco                                                              alto                                              médio
tokens_entrada                                                            257                                                347
tokens_saida                                                              250                                                224
tokens_pensamento                                 não registrado na chamada 1                                                617
tokens_totais                                                             967                                               1188
latencia_segundos                                                       3.315                    